In [111]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import plotly.graph_objects as go

from rod.helix import Helix
from rod.helix_util import HelixUtil

In [112]:
# Methods to propagate the centerline and material frames of a helix

def propagate_q(q: np.ndarray, r0: np.ndarray, n0: np.ndarray, n_sites: int, s: np.ndarray, r: np.ndarray,
                    n: np.ndarray):
        # Starting with the clamped material frame, integrate forward
        r[0, :] = r0
        n[0, :] = n0
        for i in range(1, n_sites):
            # Left hand side of interval (previous element)
            r_L = r[i - 1]
            n_L = n[i - 1]
            s_R, s_L = s[i], s[i - 1]
            s_sL = s_R - s_L
            # Twist and curvature
            tau, k_1, k_2 = q[3 * i - 3:3 * i]
            # Darboux vector and unit vector aligned with the Darboux vector
            Omega = tau * n_L[0, :] + k_1 * n_L[1, :] + k_2 * n_L[2, :]
            Omega_norm = np.linalg.norm(Omega)

            # Degenerate case: straight line/no change in material frame
            if Omega_norm < 1e-12:
                n[i] = n_L
                r[i] = r_L + n_L[0] * s_sL
                continue

            w = Omega / Omega_norm

            # Projection of vector parallel to and perpendicular to w
            n_L_par = np.dot(n_L, w)[:, np.newaxis] * w
            n_L_perp = n_L - n_L_par

            # Compute the material frame
            n_i = n_L_par + n_L_perp * np.cos(Omega_norm * s_sL) + np.cross(w, n_L_perp) * np.sin(Omega_norm * s_sL)
            n[i] = n_i

            # Compute the centerline
            n_0_parallel = n_L_par[0]
            n_0_perp = n_L_perp[0]
            r_i = (r_L + n_0_parallel * s_sL + n_0_perp * np.sin(Omega_norm * s_sL) / Omega_norm +
                   np.cross(w, n_0_perp) * (1 - np.cos(Omega_norm * s_sL)) / Omega_norm)
            r[i] = r_i
        return r, n

def propagate(helix: Helix) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes the centerline and the material frames from the generalized coordinates [q]

    n_i(s) = n_{i, L}^{Q ||} + n_{i, L}^{Q perp} cos(Omega(s - s_L^Q)) + omega \cross n_{i, L}^{Q perp} sin(Omega(s - s_L^Q))
    """
    # Centerline and material frames
    r = np.zeros((helix.n_sites, 3))
    n = np.zeros((helix.n_sites, 3, 3))
    HelixUtil.propagate_q(helix.q, helix.r0, helix.n0, helix.n_sites, helix.s, r, n)
    return r, n

In [113]:
def plot_helix(r: np.ndarray, n: np.ndarray, ax=None, show_frames=True, frame_step=2, frame_scale=0.02):
    """
    Visualizes the helix centerline and material frames with smaller frames.

    Parameters:
    - r: (n_sites, 3) Centerline positions
    - n: (n_sites, 3, 3) Material frames
    - show_frames: Whether to show material frames
    - frame_step: Step size for plotting frames (to avoid clutter)
    - frame_scale: Scaling factor for material frame vectors
    """
    if ax is None:
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(111, projection='3d')

    # Plot centerline
    ax.plot(r[:, 0], r[:, 1], r[:, 2], 'b-', label="Centerline")

    if show_frames:
        for i in range(0, len(r), frame_step):
            origin = r[i]
            for j, color in enumerate(['r', 'g', 'k']):  # x (red), y (green), z (black)
                frame_vec = n[i, j] * frame_scale  # Scale frame vectors
                ax.quiver(origin[0], origin[1], origin[2], 
                          frame_vec[0], frame_vec[1], frame_vec[2], 
                          color=color, alpha=0.8)

    # Labels and view adjustments
    # ax.set_xlabel('X')
    # ax.set_ylabel('Y')
    # ax.set_zlabel('Z')
    # ax.set_title('Helix')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    return ax 

def plot_helix_plotly(helices, show_frames=True, frame_step=2, frame_scale=0.02):
    """
    Creates an interactive visualization of multiple helices' centerlines and material frames.

    Parameters:
    - helices: Array-like of tuples (r, n), where:
      - r: (n_sites, 3) Centerline positions
      - n: (n_sites, 3, 3) Material frames
    - show_frames: Whether to show material frames
    - frame_step: Step size for plotting frames (to avoid clutter)
    - frame_scale: Scaling factor for material frame vectors
    """
    # Create figure
    fig = go.Figure()
    
    # Plot each helix
    for idx, (r, n) in enumerate(helices):
        # Plot centerline
        fig.add_trace(go.Scatter3d(
            x=r[:, 0], y=r[:, 1], z=r[:, 2],
            mode='lines',
            line=dict(width=3),
            name=f'Centerline {idx + 1}'
        ))
        
        # Plot material frames as arrows
        if show_frames:
            for i in range(0, len(r), frame_step):
                origin = r[i]
                for j, color in enumerate(['red', 'green', 'black']):  # x (red), y (green), z (black)
                    frame_vec = n[i, j] * frame_scale  # Scale frame vectors
                    fig.add_trace(go.Scatter3d(
                        x=[origin[0], origin[0] + frame_vec[0]],
                        y=[origin[1], origin[1] + frame_vec[1]],
                        z=[origin[2], origin[2] + frame_vec[2]],
                        mode='lines',
                        line=dict(color=color, width=2),
                        showlegend=False  # Avoid excessive legend entries
                    ))
    
    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
        ),
        title=dict(text='Helix Visualization', y=0.95, x=0.5, xanchor='center', yanchor='top'),
    )
    
    fig.show()


In [114]:
# Instantiating and Integrating Centerline + Material Frames for 1 Helical Segment

n_sites = 50  # Number of sites along the helix
L = 10.0  # Total length
s = np.linspace(0, L, n_sites)  # Arc length array

# Initial position: Start the curl at the origin
r0 = np.array([0.0, 0.0, 0.0])

# Initial material frame: Oriented along the curl
n0 = np.eye(3)  # Standard basis (modify if needed)

# Create a twisting and bending motion in q
twist = np.full(n_sites, np.pi / 4)
# twist = np.zeros(n_sites)
curvature_x = 3.0 * np.cos(s * np.pi) + np.random.random_sample() # Varies to form a curl
curvature_y = 0 #3.0 * np.sin(s * np.pi) + np.random.random_sample()  # Varies to form a curl

q = np.zeros((3 * n_sites,))
q[0::3] = twist
q[1::3] = curvature_x
q[2::3] = curvature_y

# Define stiffness (optional)
EI = np.ones((3 * n_sites,)) * 0.1  # Uniform stiffness

# Create helix instance
# r0 and n0 are the position and material frame of the clamped top node
helix_instance = Helix(q=q, q0=q, n_sites=n_sites, s=s, L=L, r0=r0, n0=n0, EI=EI)

# Perform one propagation step
r, n = propagate(helix_instance)

# print("Propagated centerline positions:\n", r)
# print("Propagated material frames:\n", n)

# Generate and propagate a helix (from previous code)
r, n = HelixUtil.propagate(helix_instance)
print(r.shape)

# Plot the helix with small material frames
# plot_helix_plotly([(r, n)], frame_scale=0.2, show_frames=True)
# plot_helix(r, n, show_frames=False, frame_scale=0.2)

(50, 3)


### Creating SDF by Interpolating Between Follicle Starting Points

In [115]:
## Identifying Starting Points

import plotly
import plotly.graph_objects as go

pos, edges = [], []
with open("normals_one_seg.obj", 'r') as f:
    for line in f:
        if line[0] == 'v':
            pos.append(list(map(float, line.split()[1:])))
        elif line[0] == 'l':
            edges.append(list(map(int, line.split()[1:])))
pos = np.array(pos)
edges = np.array(edges) - 1

# Convert y up and center the positions to origin
pos = pos[:, [0, 2, 1]]
pos -= np.mean(pos, axis=0)

# Create strands starts
start_strands = []
for i1, i2 in edges:
    start_strands.append((pos[i1], pos[i2]))
start_strands = np.array(start_strands)
start_strands = start_strands[:2000] # 2000 strands. start & end pts per strand. 3 coords per point

# Create initial positions and directions
r0 = start_strands[:, 0]
print(r0)

fig = go.Figure(data=[go.Scatter3d(
    x=r0[:, 0], 
    y=r0[:, 1], 
    z=r0[:, 2], 
    mode='markers', 
    marker=dict(size=2)
)])
fig.show()

[[-0.08275487  0.08305151  0.00852626]
 [-0.08394087  0.07976051  0.00651424]
 [-0.08149487  0.0846645   0.01363032]
 ...
 [ 0.05732513  0.12570751  0.00526326]
 [ 0.06013613  0.12300651  0.00486129]
 [ 0.04641213  0.13376251  0.00992732]]


In [116]:
import numpy as np
from scipy.spatial.transform import Rotation as R

r0_mean = np.mean(r0, axis=0)
r0_std = np.std(r0, axis=0)  # Avoids large numerical values
r0_scaled = (r0 - r0_mean) / r0_std  # Normalize
r0_scaled += 1e-6 * np.random.randn(*r0_scaled.shape) # jitter to avoid singular matrix

# Step 1: Fit an approximate ellipsoid
center = np.mean(r0_scaled, axis=0)  # Compute the mean
cov = np.cov(r0_scaled.T)            # Compute the covariance matrix
U, S, _ = np.linalg.svd(cov)          # SVD to get principal axes

radii = np.sqrt(S)  # Approximate radii of the ellipsoid

# Step 2: Generate uniform points on a unit sphere
num_points = 5000  # Adjust based on desired density
phi = np.arccos(1 - 2 * np.linspace(0, 1, num_points))
theta = np.pi * (1 + 5**0.5) * np.arange(num_points)  # Golden ratio spiral

x = np.sin(phi) * np.cos(theta)
y = np.sin(phi) * np.sin(theta)
z = np.cos(phi)
unit_sphere_points = np.vstack((x, y, z)).T  # Shape (num_points, 3)

# Step 3: Transform unit sphere to ellipsoid
ellipsoid_points = unit_sphere_points * radii*1.7  # Scale by radii
ellipsoid_points = ellipsoid_points @ U.T  # Rotate to match original orientation
ellipsoid_points += center  # Translate to match original data
ellipsoid_points[:, 1] - 0.25
ellipsoid_points[:, 2] - 0.1

fig = go.Figure()

# Add the ellipsoid points in one color (e.g., blue)
fig.add_trace(go.Scatter3d(
    x=ellipsoid_points[:, 0], 
    y=ellipsoid_points[:, 1], 
    z=ellipsoid_points[:, 2], 
    mode='markers', 
    marker=dict(size=2, color='blue'),
    name="Ellipsoid"
))

# Add the original r0_scaled points in another color (e.g., red)
fig.add_trace(go.Scatter3d(
    x=r0_scaled[:, 0], 
    y=r0_scaled[:, 1], 
    z=r0_scaled[:, 2], 
    mode='markers', 
    marker=dict(size=2, color='red'),
    name="Original Points"
))

fig.update_layout(title="Ellipsoid Fit vs Original Data", showlegend=True)
fig.show()

In [117]:
## Doing Radial Basis Function Interpolation on Scalp Points -

from scipy.interpolate import RBFInterpolator

# Creating training data
offset = 0.7 
ellipsoid_center = np.mean(ellipsoid_points, axis=0)
inner_points = ellipsoid_points * (1 - offset) + ellipsoid_center * offset  # Move inward
outer_points = ellipsoid_points * (1 + offset) - ellipsoid_center * offset  # Move outward

sdf_values = np.concatenate([
    np.zeros(len(ellipsoid_points)),  # Surface
    -offset * np.ones(len(inner_points)),  # Inside
    offset * np.ones(len(outer_points))  # Outside
])

# Combine all points
sdf_points = np.vstack((ellipsoid_points, inner_points, outer_points))

rbf = RBFInterpolator(
    sdf_points, sdf_values, 
    kernel='multiquadric',
    epsilon=0.1,  # regularization
)
# Define SDF function
def scalp_sdf(p):
    # p_scaled = (p - r0_mean) / r0_std  # Apply same scaling
    return rbf(p)

In [118]:
n_sites = 50  # Number of sites along the helix
L = 10.0  # Total length
s = np.linspace(0, L, n_sites)  # Arc length array

# Initial position: Start the curl at the origin
r0 = np.array([0.0, 0.0, 0.0])

# Initial material frame: Oriented along the curl
n0 = np.eye(3)  # Standard basis (modify if needed)

# Create a twisting and bending motion in q
twist = np.full(n_sites, np.pi / 4)
# twist = np.zeros(n_sites)
curvature_x = 3.0 * np.cos(s * np.pi) + np.random.random_sample() # Varies to form a curl
curvature_y = 0 #3.0 * np.sin(s * np.pi) + np.random.random_sample()  # Varies to form a curl

q = np.zeros((3 * n_sites,))
q[0::3] = twist
q[1::3] = curvature_x
q[2::3] = curvature_y

# Define stiffness (optional)
EI = np.ones((3 * n_sites,)) * 0.1  # Uniform stiffness

# Create helix instance
# r0 and n0 are the position and material frame of the clamped top node
helix_instance = Helix(q=q, q0=q, n_sites=n_sites, s=s, L=L, r0=r0, n0=n0, EI=EI)

In [119]:
def sdf_gradient(p, eps=1e-4):
    grad = np.zeros((p.shape[0], 3))
    for i in range(3):
        dp = np.zeros(3)
        dp[i] = eps
        grad = (scalp_sdf(p + dp) - scalp_sdf(p - dp)) / (2 * eps)
    return grad / np.linalg.norm(grad)  # Normalize

# Testing restoring force calculation
grad = np.zeros(3 * (helix_instance.n_sites - 1))
eps = 1e-6
k_collision = 1e4

r, _ = HelixUtil.propagate(helix_instance)
d = scalp_sdf(r)  # Expected shape: (n_sites,)

grad = grad.reshape((helix_instance.n_sites - 1, 3))  # Reshape for clarity

for j in range(helix_instance.n_sites - 1):
    if d[j] < 0:  # Inside scalp
        contact_force = -k_collision * d[j] * sdf_gradient(r)[j]  # Ensure correct shape
        grad[j] += contact_force  # Element-wise vector addition

grad = grad.flatten()  # Restore original shape if needed
print(grad)

[  -17.12696857   -17.12696857   -17.12696857   -67.05792492
   -67.05792492   -67.05792492  -191.79471487  -191.79471487
  -191.79471487  -324.2765587   -324.2765587   -324.2765587
  -399.30476993  -399.30476993  -399.30476993  -379.64117102
  -379.64117102  -379.64117102  -321.21017605  -321.21017605
  -321.21017605  -243.67835368  -243.67835368  -243.67835368
  -116.99784073  -116.99784073  -116.99784073     0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.
     0.             0.             0.             0.

In [123]:
# 0 level-set
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import RBFInterpolator
from skimage.measure import marching_cubes

# Create a 3D grid for SDF evaluation
res = 50  # Grid resolution (increase for more detail)
x = np.linspace(r0_scaled[:,0].min()-1, r0_scaled[:,0].max()+1, res)
y = np.linspace(r0_scaled[:,1].min()-1, r0_scaled[:,1].max()+1, res)
z = np.linspace(r0_scaled[:,2].min()-1, r0_scaled[:,2].max()+1, res)
X, Y, Z = np.meshgrid(x, y, z)
grid_points = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))

# Evaluate SDF at each grid point
sdf_values = scalp_sdf(grid_points).reshape(X.shape)

# Extract the zero level-set using Marching Cubes
verts, faces, _, _ = marching_cubes(sdf_values, level=0, spacing=(x[1] - x[0], y[1] - y[0], z[1] - z[0]))

# Convert to a Plotly-compatible mesh
fig = go.Figure(data=[go.Mesh3d(
    x=verts[:, 0], 
    y=verts[:, 1], 
    z=verts[:, 2], 
    i=faces[:, 0], 
    j=faces[:, 1], 
    k=faces[:, 2], 
    color='lightblue', 
    opacity=0.5
)])
fig.update_layout(title="0 Level Sets of SDF", showlegend=True)
fig.show()


### Computing Restoring Force for Strands Inside Head

In [121]:
def sdf_gradient(p, eps=1e-4):
    grad = np.zeros(3)
    for i in range(3):
        dp = np.zeros(3)
        dp[i] = eps
        grad[i] = (scalp_sdf(p + dp) - scalp_sdf(p - dp)) / (2 * eps)
    return grad / np.linalg.norm(grad)  # Normalize

def compute_gen_force(helix: Helix, g: float, rhoS: float, seed: int, sdf_func) -> np.ndarray:
    """
    Computes the generalized force including gravity and scalp contact forces using SDF.
    """
    grad = np.zeros(3 * (helix.n_sites - 1))
    eps = 1e-6
    k_collision = 1e4  # Stiffness for head-hair contact force

    q_free = helix.q.copy()[3:]
    for i in range(3 * (helix.n_sites - 1)):
        q_plus = q_free.copy()
        q_plus[i] += eps
        helix.q = np.concatenate([helix.q[:3], q_plus])
        r_plus, _ = HelixUtil.propagate(helix)
        U_g_plus = HelixUtil.compute_gen_potential_pos(helix, r_plus, g, rhoS, seed)

        q_minus = q_free.copy()
        q_minus[i] -= eps
        helix.q = np.concatenate([helix.q[:3], q_minus])
        r_minus, _ = HelixUtil.propagate(helix)
        U_g_minus = HelixUtil.compute_gen_potential_pos(helix, r_minus, g, rhoS, seed)

        # Finite difference gravity force
        grad[i] = (U_g_plus - U_g_minus) / (2 * eps)

    # Compute contact forces from SDF
    for i, p in enumerate(helix.r):  # Iterate over hair site positions
        d = sdf_func(p)  # Compute SDF value
        if d < 0:  # Inside scalp
            sdf_grad = sdf_gradient(p, sdf_func)  # Outward direction
            contact_force = -k_collision * d * sdf_grad
            grad[3 * i: 3 * i + 3] += contact_force  # Apply force to corresponding DOFs

    return -grad

## Visualization with Smaller Ellipse (scaled 1.4), Hair Length L = 0.5, Offset for SDF 0.2

In [122]:
import pickle

with open("evolved_helices_contacts.pkl", "rb") as f:
    evolved_helices = pickle.load(f)

helix_params = []
for helix in evolved_helices[:200]:
    r, n = HelixUtil.propagate(helix)
    helix_params.append((r, n))

plot_helix_plotly(helix_params, show_frames=False, frame_scale=0.2)

## Visualization with Smaller Ellipse (scaled 1.7), Hair Length L = 0.5, Offset for SDF 0.7

In [124]:
with open("evolved_helices_contacts_bigger.pkl", "rb") as f:
    evolved_helices = pickle.load(f)

helix_params = []
for helix in evolved_helices[:200]:
    r, n = HelixUtil.propagate(helix)
    helix_params.append((r, n))

plot_helix_plotly(helix_params, show_frames=False, frame_scale=0.2)

In [125]:
with open("evolved_helices_contacts_smallo_bige.pkl", "rb") as f:
    evolved_helices = pickle.load(f)

helix_params = []
for helix in evolved_helices[:200]:
    r, n = HelixUtil.propagate(helix)
    helix_params.append((r, n))

plot_helix_plotly(helix_params, show_frames=False, frame_scale=0.2)